## **Preparing the PyTorch Dataset and DataLoaders**


Since patches has been saved and a patch_df.csv created, I need to build a PyTorch pipeline that can:

 - Load each patch from disk using the file path.
 - Apply optional data augmentation (for training only).
 - Convert the image to a normalized tensor (expected by ResNet).
 - Return a (tensor, label) pair for training.
 - We’ll also set up PyTorch DataLoaders for each split (train, val, test).

In [4]:
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd

# Custom Dataset
class BreakHisDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.labels = {'benign': 0, 'malignant': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'path']
        label = self.labels[self.df.loc[idx, 'label']]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

# Transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load patch_df
patch_df = pd.read_csv('/content/drive/MyDrive/BreakHis/BreaKHis_v1/patches/patch_df.csv')

# Create datasets
train_dataset = BreakHisDataset(patch_df[patch_df['split']=='train'], transform=train_transform)
val_dataset = BreakHisDataset(patch_df[patch_df['split']=='val'], transform=val_test_transform)
test_dataset = BreakHisDataset(patch_df[patch_df['split']=='test'], transform=val_test_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"✅ DataLoaders ready!")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

✅ DataLoaders ready!
Train batches: 1042, Val batches: 189, Test batches: 254


## **Model Setup**

### What It Does
 - Detects GPU/CPU and sets device
 - Loads ResNet50 pre-trained on ImageNet (already knows basic image features)
 - Replaces final layer: 1000 classes → 2 classes (benign/malignant)
 - Moves model to GPU for faster training

### Why ResNet50?
Pre-trained on millions of images, so it already understands edges, textures, and patterns. We just fine-tune it for breast cancer.

In [6]:
import torch
import torch.nn as nn
from torchvision import models

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load pretrained ResNet50
model = models.resnet50(pretrained=True)

# Modify final layer for binary classification
model.fc = nn.Linear(model.fc.in_features, 2)

# Move to device
model = model.to(device)

print("✅ ResNet50 model loaded and ready!")
print(f"Model output: 2 classes (benign=0, malignant=1)")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 186MB/s]


✅ ResNet50 model loaded and ready!
Model output: 2 classes (benign=0, malignant=1)


## **Weighted Loss Function**

### The Problem
We have fewer benign samples than malignant. Without correction, the model could ignore benign cases and still get high accuracy.

### What It Does
 - Counts training samples per class (e.g., 5000 benign, 2000 malignant)
 - Calculates inverse frequency weights:
     * Malignant (rare) → higher weight (e.g., 1.75)
     * Benign (common) → lower weight (e.g., 0.70)
 - Creates weighted CrossEntropyLoss

### Outcome
Malignant mistakes now penalize the model 2.5x more than Benign mistakes, forcing it to learn both classes equally.

In [7]:
import pandas as pd

# Load patch_df to calculate class weights
patch_df = pd.read_csv('/content/drive/MyDrive/BreakHis/BreaKHis_v1/patches/patch_df.csv')
train_df = patch_df[patch_df['split'] == 'train']

# Count samples per class
label_counts = train_df['label'].value_counts()
print("Training set class distribution:")
print(label_counts)

# Calculate class weights (inverse frequency)
total_samples = len(train_df)
benign_count = label_counts['benign']
malignant_count = label_counts['malignant']

weight_benign = total_samples / (2 * benign_count)
weight_malignant = total_samples / (2 * malignant_count)

# Create weight tensor [weight_for_class_0, weight_for_class_1]
class_weights = torch.tensor([weight_benign, weight_malignant], dtype=torch.float).to(device)

print(f"\nClass weights: benign={weight_benign:.4f}, malignant={weight_malignant:.4f}")

# Create weighted loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)

print("✅ Weighted CrossEntropyLoss created!")

Training set class distribution:
label
malignant    22446
benign       10872
Name: count, dtype: int64

Class weights: benign=1.5323, malignant=0.7422
✅ Weighted CrossEntropyLoss created!


## **Optimizer Setup**

### What It Does

Creates Adam optimizer with learning rate 0.0001

### Why Adam?
 - Adapts learning rate automatically per parameter
 - Works well for fine-tuning pre-trained models
 - Industry standard
### Why lr=0.0001?
- We're fine-tuning (not training from scratch)
- Small updates preserve pre-trained ImageNet features
 - Too high (0.01) destroys useful features
 - Too low (0.000001) trains too slowly

### How It Works
Each training step: Model predicts → Calculate loss → Compute gradients → Optimizer updates weights

In [8]:
import torch.optim as optim

# Setup optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("✅ Adam optimizer configured with lr=0.0001")
print("\nReady for training!")

✅ Adam optimizer configured with lr=0.0001

Ready for training!


## **Training Function**

This function trains the model for one epoch. It:
 - Sets model to training mode (enables dropout, batch norm updates)
 - Loops through all training batches
 - Performs forward pass, calculates loss, backpropagation, and updates weights
 - Tracks running loss and accuracy

In [11]:
from tqdm import tqdm

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return running_loss / len(loader), 100 * correct / total

## **Validation Function**

### This function evaluates the model on validation set. It:
 - Sets model to evaluation mode (disables dropout, batch norm in eval mode)
 - Uses torch.no_grad() to disable gradient calculation (saves memory)
 - Loops through validation batches and calculates loss and accuracy
 - No weight updates happen here

In [12]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return running_loss / len(loader), 100 * correct / total

## **Early Stopping Class**

###Monitors validation loss and stops training if no improvement for patience epochs.

 - Tracks best validation loss
 - Counts epochs without improvement
 - Signals when to stop training

In [13]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

## **Training Loop with Checkpointing**
This is the main training loop. It:

 - Trains for specified epochs
 - Validates after each epoch
 - Saves checkpoint after EVERY epoch (so you can resume if interrupted)
 - Saves best model based on validation accuracy
 - Implements early stopping if validation loss stops improving

In [14]:
import os

# Training configuration
num_epochs = 30
checkpoint_dir = '/content/drive/MyDrive/BreakHis/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Initialize early stopping
early_stopping = EarlyStopping(patience=5, min_delta=0.001)

# Track best accuracy
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print('='*50)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Print metrics
    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Save checkpoint every epoch
    checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth')
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'history': history
    }, checkpoint_path)
    print(f"💾 Checkpoint saved: epoch_{epoch+1}.pth")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_path = os.path.join(checkpoint_dir, 'best_model.pth')
        torch.save(model.state_dict(), best_model_path)
        print(f"⭐ Best model saved! Val Acc: {val_acc:.2f}%")

    # Early stopping check
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print(f"\n🛑 Early stopping triggered at epoch {epoch+1}")
        break

print(f"\n🎉 Training complete!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")


Epoch 1/30


Validation: 100%|██████████| 189/189 [00:53<00:00,  3.54it/s]



Train Loss: 0.0386 | Train Acc: 98.58%
Val Loss: 1.0308 | Val Acc: 90.02%
💾 Checkpoint saved: epoch_1.pth
⭐ Best model saved! Val Acc: 90.02%

Epoch 2/30


Validation: 100%|██████████| 189/189 [00:33<00:00,  5.63it/s]



Train Loss: 0.0108 | Train Acc: 99.67%
Val Loss: 0.6195 | Val Acc: 88.44%
💾 Checkpoint saved: epoch_2.pth

Epoch 3/30


...


Train Loss: 0.0002 | Train Acc: 100.00%
Val Loss: 1.4992 | Val Acc: 80.02%
💾 Checkpoint saved: epoch_7.pth

Epoch 8/30


Validation: 100%|██████████| 189/189 [00:33<00:00,  5.66it/s]



Train Loss: 0.0124 | Train Acc: 99.62%
Val Loss: 1.0654 | Val Acc: 82.65%
💾 Checkpoint saved: epoch_8.pth

Epoch 9/30


Validation: 100%|██████████| 189/189 [00:33<00:00,  5.71it/s]



Train Loss: 0.0055 | Train Acc: 99.83%
Val Loss: 0.9341 | Val Acc: 84.23%
💾 Checkpoint saved: epoch_9.pth

🛑 Early stopping triggered at epoch 9

🎉 Training complete!
Best validation accuracy: 90.02%


In [3]:
from tqdm import tqdm

In [4]:
# First, reload the model
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recreate model architecture
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)

# Load best weights
best_model_path = '/content/drive/MyDrive/BreakHis/checkpoints/best_model.pth'
model.load_state_dict(torch.load(best_model_path))
model = model.to(device)

print("✅ Best model loaded!")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Best model loaded!


**Purpose:** Treats each patch as an independent test sample and evaluates accuracy on individual patches.

**How it works:**

  1. Sets model to evaluation mode (model.eval()) - disables dropout and batch normalization training behavior
  2. Iterates through the test_loader, which contains individual patch images
  3. For each batch:
      - Moves images to GPU/CPU
      - Gets model predictions (forward pass only, no gradients)
      - Extracts the predicted class (0=benign, 1=malignant) using torch.max
      - Stores predictions and true labels in lists
  4. After processing all patches, calculates metrics:
      - Accuracy: Percentage of correctly classified patches
      - Precision: Of all patches predicted as malignant, how many actually were malignant
      - Recall: Of all truly malignant patches, how many did we correctly identify
      - F1-Score: Harmonic mean of precision and recall
      - Confusion Matrix: Shows true positives, false positives, true negatives, false negatives

**Limitation:** This approach treats patches independently, ignoring that multiple patches come from the same slide. A single ambiguous patch from a malignant tumor might look benign, leading to individual misclassifications even though the overall tumor is clearly malignant.

In [5]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np

def evaluate_image_level(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Testing"):
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    cm = confusion_matrix(all_labels, all_preds)

    print("\n" + "="*50)
    print("IMAGE-LEVEL TEST RESULTS")
    print("="*50)
    print(f"Accuracy:  {accuracy*100:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"              Benign  Malignant")
    print(f"Actual Benign    {cm[0,0]:5d}     {cm[0,1]:5d}")
    print(f"     Malignant   {cm[1,0]:5d}     {cm[1,1]:5d}")

    return all_preds, all_labels

# Run image-level evaluation
img_preds, img_labels = evaluate_image_level(model, test_loader, device)

Testing: 100%|██████████| 254/254 [01:07<00:00,  3.78it/s]



IMAGE-LEVEL TEST RESULTS
Accuracy:  91.52%
Precision: 0.9128
Recall:    0.9586
F1-Score:  0.9351

Confusion Matrix:
                Predicted
              Benign  Malignant
Actual Benign     2467       473
     Malignant     214      4952


## Slide-Level Evaluation (What the code does)

**Purpose:** Aggregates predictions from all patches belonging to the same slide and makes one diagnosis per slide. This mimics how pathologists actually work - they examine multiple regions before making a diagnosis.

**How it works:**

  1. Gets predictions for ALL test patches (same as image-level)
  2. Additionally stores probability scores for each class (using torch.softmax)
  3. Creates a copy of the test dataframe and adds three new columns:
      - prediction: The predicted class (0 or 1)
      - prob_benign: Probability the patch is benign
      - prob_malignant: Probability the patch is malignant
  4. Groups all patches by their slide_id (all patches from the same slide are grouped together)
  5. For each slide:
      - Takes all predictions from that slide's patches
      - Uses majority vote (.mode()) - whichever class appears most often becomes the slide's prediction
      - Gets the true label (same for all patches from that slide)
  6. Creates a new dataframe where each row is one slide (not one patch)
  7. Calculates the same metrics (accuracy, precision, recall, F1, confusion matrix) but at the slide level

**Why this is better:**

  - More clinically realistic - pathologists don't diagnose from one tiny region
  - More robust - a few misclassified patches won't override the overall pattern
  - Reduces noise from ambiguous or low-quality individual patches
  - This is what we discussed earlier about preventing data leakage - slides in test set are completely unseen, and we evaluate the model's ability to diagnose entirely new patients

In [6]:
import pandas as pd
from tqdm import tqdm

def evaluate_slide_level(model, test_df, device, batch_size=32):
    model.eval()

    # Get predictions for all test images
    all_preds = []
    all_probs = []

    test_dataset = BreakHisDataset(test_df, transform=val_test_transform)
    test_loader_full = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    with torch.no_grad():
        for images, _ in tqdm(test_loader_full, desc="Getting predictions"):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # Add predictions to dataframe
    test_df_copy = test_df.copy()
    test_df_copy['prediction'] = all_preds
    test_df_copy['prob_benign'] = [p[0] for p in all_probs]
    test_df_copy['prob_malignant'] = [p[1] for p in all_probs]

    # Aggregate by slide_id (majority vote)
    slide_results = []
    for slide_id in test_df_copy['slide_id'].unique():
        slide_data = test_df_copy[test_df_copy['slide_id'] == slide_id]

        # Majority vote
        slide_pred = slide_data['prediction'].mode()[0]

        # True label (same for all images in slide)
        slide_label = 0 if slide_data['label'].iloc[0] == 'benign' else 1

        slide_results.append({
            'slide_id': slide_id,
            'true_label': slide_label,
            'prediction': slide_pred
        })

    slide_df = pd.DataFrame(slide_results)

    # Calculate metrics
    accuracy = accuracy_score(slide_df['true_label'], slide_df['prediction'])
    precision, recall, f1, _ = precision_recall_fscore_support(
        slide_df['true_label'], slide_df['prediction'], average='binary'
    )
    cm = confusion_matrix(slide_df['true_label'], slide_df['prediction'])

    print("\n" + "="*50)
    print("SLIDE-LEVEL TEST RESULTS")
    print("="*50)
    print(f"Total slides: {len(slide_df)}")
    print(f"Accuracy:  {accuracy*100:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"              Benign  Malignant")
    print(f"Actual Benign    {cm[0,0]:5d}     {cm[0,1]:5d}")
    print(f"     Malignant   {cm[1,0]:5d}     {cm[1,1]:5d}")

    return slide_df

# Load test data
patch_df = pd.read_csv('/content/drive/MyDrive/BreakHis/BreaKHis_v1/patches/patch_df.csv')
test_df = patch_df[patch_df['split'] == 'test']

# Run slide-level evaluation
slide_results = evaluate_slide_level(model, test_df, device)

Getting predictions: 100%|██████████| 254/254 [00:47<00:00,  5.35it/s]


SLIDE-LEVEL TEST RESULTS
Total slides: 13
Accuracy:  92.31%
Precision: 0.8889
Recall:    1.0000
F1-Score:  0.9412

Confusion Matrix:
                Predicted
              Benign  Malignant
Actual Benign        4         1
     Malignant       0         8
